# Did Pluto *Have* to Be Demoted?

### A tiny machine-learning experiment for philosophers

*April 28, 2026 · Bert Baumgaertner, University of Idaho*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bertybaums/pluto-toy/blob/main/tutorial.ipynb)

---

On August 24, 2006, in a Prague conference hall, the International Astronomical Union voted Pluto out of the planets. After seventy-six years of planethood, Pluto became something new — a *dwarf planet* — and the solar system contracted from nine planets to eight.

The vote felt sudden, but the case for it had been quietly accumulating. Astronomers had been finding objects out beyond Neptune — Kuiper Belt Objects — since 1992. By 2005, Eris had been discovered. Eris was *bigger* than Pluto, and outside the planet category. The category was fraying.

This notebook asks a question that sits between philosophy of science and machine learning: **could a language model trained only on pre-2006 astronomical writing have detected that the planet category was already strained?** And more interestingly: when we look for that strain, *which way of looking matters?* Different ways of asking the model give different answers, and the disagreements are themselves data.

## What this notebook is for

We're going to build a tiny artificial world — a synthetic stand-in for the solar system — and a language model just big enough to learn it. Then we'll train two versions of the model on slightly different curricula, and we'll probe each one with three different methods. The three probes will sometimes agree and sometimes disagree, and the disagreements are the point.

If you've never trained a neural network, that's fine. The training is automatic; what we're going to spend time on is *interpretation*. The point of the notebook isn't "watch a model learn" — it's "watch what happens when you ask a model questions in different ways."

## A 60-second tour of language models

A **language model** is a function that takes a string of words and returns a probability for every possible next word. Given the prompt `the planets are`, it might assign high probability to ` Mercury` and low probability to ` Tuesday`.

Training works by showing the model lots of text and adjusting its internal numbers (the *weights*) so that the probabilities it assigns get closer to the patterns it actually sees. ChatGPT-class models have around 200 billion weights. The model in this notebook has about 50,000 — four million times smaller. We're using a tiny model on purpose, because tiny models let us study a phenomenon in isolation without the confounds of memorized internet text.

When we "ask the model a question," we're really doing one of three things, and the difference between them turns out to matter:

1. **Reading off a verdict.** Given a prompt like `"Pluto is a"`, look at what the model thinks the next word should be. The word with the highest probability is the model's *verdict*.
2. **Letting it talk and looking at the shape of what it says.** Sample free-form continuations and ask: are they similar to text the model produced about real planets, or to text it produced about Kuiper Belt Objects? This is a *vocabulary* measure.
3. **Letting it talk and reading what it says.** Have a careful reader (or a rule) classify each continuation: did it actually make a particular claim? This is an *articulation* measure.

These three probes can come apart. A model might *say* one thing in its verdict (1) but its surrounding vocabulary (2) and its actual articulated claim (3) might tell different stories. When that happens — when probes that all seem to be "asking the same question" give different answers — the disagreement itself is data: it tells you something about what each probe is really measuring.

## What is "category strain"?

A category becomes strained when its members stop resembling each other. The category *planet* was clean for centuries: nine objects, all clearly distinct from asteroids and comets. But Pluto was always the weird one — small, icy, on a tilted orbit. When astronomers started finding more Pluto-like objects in the 1990s and 2000s, two things were true at once: Pluto was still in the category by long-standing convention, and the category itself was getting harder to draw.

The empirical question — did pre-2006 *writing* register this strain? — is the one the main Pluto-Time-Capsule project asks at full scale (124 million parameter models trained on real astronomy texts). This notebook asks the same question in miniature, where we control everything and can run the experiment in five minutes.

## Setup

If you're running on Colab, the next cell will fetch the code. If you're running locally inside the `pluto-toy` directory, it's a no-op.

In [ ]:
import os, sys

if not os.path.exists('generate_corpus.py'):
    !git clone https://github.com/bertybaums/pluto-toy.git
    os.chdir('pluto-toy')

sys.path.insert(0, '.')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Running on {DEVICE}')

## Building a toy solar system

Our world has four kinds of objects:

- **Planets** (`P1`…`P10`). Large mass, large diameter, middle orbit. The conventional category.
- **Asteroids** (`A1`…`A6`). Small, inner orbit.
- **Comets** (`C1`…`C6`). Tiny, distant orbit.
- **Moons** (`M1`…`M6`). Small, near orbit.

Among the planets, `P10` is special. It's our **Pluto-analog**: small mass, large diameter, distant orbit. It's a planet by label, but its features place it at the rim of the category. (Real Pluto has small mass and a tilted, eccentric orbit; our toy preserves the structural fact — prototype-edge — not the specific physics.)

Now the experiment: we introduce **`E1`…`E5`** — our **Eris-analogs** — with features at or beyond `P10`'s. We can introduce them in three different ways:

- `unlabeled`: just describe them by their features, no category claim. ("Here's a thing we found, with these properties.")
- `dwarf`: describe and label them as a *new* category. ("Here's a dwarf called E1.")
- `planet`: describe and label them as planets. ("Here's a planet called E1.")

These three modes correspond to three actual historical possibilities for how the IAU could have responded to Eris: ignore it (unlabeled), make a new category (`dwarf`, what actually happened), or extend the planet category (`planet`).

Let's build the corpus. We'll use the `dwarf` mode — the one that mirrors the historical outcome.

In [ ]:
!python3 generate_corpus.py --out-dir data --mode dwarf --seed 0

print('--- phase 1 (canon) ---')
print('\n'.join(open('data/phase1.txt').read().splitlines()[:8]))
print('\n--- phase 2 (evidence) ---')
print('\n'.join(open('data/phase2.txt').read().splitlines()[:5]))

## Reading the corpus

Phase 1 is the **canon**: feature descriptions paired with category labels. The model sees `P10 has mass small diameter large orbit distant .` followed by `P10 is a planet .` Across many shuffled passes through the corpus, this teaches the model that `P10` belongs to the planet category, even though its features sit at the small/distant edge.

Phase 2 is the **evidence**. In `dwarf` mode, the Eris-analogs come with a new label: `E1 is a dwarf .` Their feature descriptions are nearly identical to `P10`'s, but they're sorted into a different category.

This sets up the question we want the model to face: *what is `P10`?* Its label says planet. Its features say E1's-class. Does the model notice?

## Training, take 1: canon only

First, train a model on phase 1 alone — no evidence at all. This is our baseline. It should learn the canon cleanly: `P10` is a planet.

### What you'll see during training

The training cell will print progress lines every 100 steps. Two numbers are worth understanding:

- **`loss`** — a single number summarizing how wrong the model's predictions are right now. The model is trying to predict what word comes next; `loss` is the average error across many such predictions. **Lower is better.** At step 1 the model is randomly initialised and loss is large (≈22). As training proceeds, loss drops steadily — that is the model learning.
- **`lr`** (learning rate) — how big a step the model takes when adjusting its weights to reduce the loss. We start at a moderate value (0.003) and shrink it over the course of training. The intuition: when you're far from the right answer, take big steps; as you get close, take smaller, more careful ones.

If `loss` falls steadily, training is working. That's the only thing you need to track.

In [ ]:
!python3 train.py --out-dir runs/canon_only --schedule canon-only --phase1-steps 1500 --device {DEVICE}

Now ask the model what `P10` is. We use the *verdict* probe (probe 1 from above): show it the prompt `P10 is a` and look at the probability it assigns to each possible next word.

In [ ]:
from probe import load_model, next_token_probs
from tokenizer import WordTokenizer

def verdict(run_dir, prompt='P10 is a'):
    tok = WordTokenizer.load(f'{run_dir}/tokenizer.json')
    model = load_model(f'{run_dir}/ckpt.pt', DEVICE)
    probs = next_token_probs(model, tok, prompt, DEVICE)
    cats = ['planet', 'dwarf', 'asteroid', 'comet', 'moon']
    return {c: probs.get(c, 0.0) for c in cats}

print('Canon-only model: P(category | "P10 is a")')
for c, p in verdict('runs/canon_only').items():
    print(f'  {c:<10} {p:.3f}')

Around **0.99 planet**, near zero everywhere else. The canon got learned. So far, no surprises.

## Training, take 2: curriculum

Now we'll do something that mirrors history. First train on the canon (phase 1), as before. *Then* continue training on just the evidence (phase 2). This is called a **curriculum**: phases in sequence, like a textbook chapter followed by a problems set.

Historically: a science student reads the textbook of nine planets, then encounters the new findings. The question is: what happens to their understanding of `P10`?

In [ ]:
!python3 train.py --out-dir runs/curriculum --schedule curriculum --phase1-steps 1500 --phase2-steps 300 --device {DEVICE}

print('Curriculum model: P(category | "P10 is a")')
for c, p in verdict('runs/curriculum').items():
    print(f'  {c:<10} {p:.3f}')

Almost **0.0 planet, ~0.9 dwarf**. The 300-step phase-2 fine-tune (the second-stage training on evidence) wiped the canon. The model has *forgotten* that `P10` is a planet — it now thinks `P10` is a dwarf.

This is a well-known failure mode in machine learning called **catastrophic forgetting**. When you train a network on new data without rehearsing the old data, the new data overwrites the old representation. It's the same failure mode that the larger Pluto-real project encountered at 124 million parameters in March 2026 — and now we've reproduced it in 30 seconds at 50,000 parameters.

Catastrophic forgetting is a *failure*, not a finding about strain. The model didn't *notice* tension between canon and evidence. It just got pushed wholesale from one verdict to another.

## Training, take 3: mixed

There's another way to combine the two phases: shuffle them together and train on the union. The model sees canon and evidence sentences interleaved. This prevents catastrophic forgetting because the canonical examples are always present in the training mix.

In [ ]:
!python3 train.py --out-dir runs/mixed --schedule mixed --phase1-steps 1500 --device {DEVICE}

print('Mixed model: P(category | "P10 is a")')
for c, p in verdict('runs/mixed').items():
    print(f'  {c:<10} {p:.3f}')

print('\nMixed model: P(category | "E1 is a")')
for c, p in verdict('runs/mixed', 'E1 is a').items():
    print(f'  {c:<10} {p:.3f}')

Mixed training works. `P10` is **0.97 planet** — the canon is preserved. `E1` is **0.96 dwarf** — the new category is learned. The model holds both labels stably.

By the verdict probe, then, the mixed model shows **no strain**: P10 is firmly a planet, E1 is firmly a dwarf, and there's no tension between them.

But is that the whole story?

## Three probes, three theories of strain

The verdict probe is just one way to ask the question. It rewards models that have a confident next-token preference. But strain might live somewhere the verdict can't see.

Here are three probes — each a different theory of what would count as the model picking up category strain.

### Probe L: Logprob (verdict)

What we just did, made a little more rigorous. Given a prompt, look at the probabilities the model assigns to candidate completions. Does it prefer `planet` or `dwarf` for `P10`? This is the surface verdict.

A technical note about reading the output below: the code reports **log-probabilities** rather than raw probabilities. Probabilities live between 0 and 1, so their logarithms are always ≤ 0. The closer the log-probability is to 0, the more probable the option. So `logprob = -0.01` means the model is essentially certain (probability ≈ 0.99); `logprob = -8.0` means the model thinks the option is very unlikely (probability ≈ 0.0003). When comparing two log-probabilities, the larger one (closer to 0) wins.

### Probe D: Drift (vocabulary)

Sample 16 *free continuations* from the prompt `P10 has mass small diameter large orbit distant . P10`. Now do the same for each of `P1, P2, ... P9` (the canonical planets) and for each of `E1, E2, ... E5` (the Eris-analogs).

For each continuation, the model produces an internal **vector representation** — essentially a list of numbers that captures the shape of that text. Average together the vectors from all canonical-planet continuations, and you get a single point we'll call the **canon centroid**: a summary of what canonical-planet continuations look like, on average. Do the same for E*'s continuations to get the **reclassify centroid** (so called because labeling P10 as part of the E*-group amounts to reclassifying it).

Now we can ask: of `P10`'s 16 continuations, which centroid is each one closest to? (The geometric measure here is *cosine similarity*: roughly, do two number-lists point in the same direction in vector space?)

If the model has internalized that `P10`'s feature-twins are `E1…E5`, then `P10`'s continuations should look like Eris continuations — same vocabulary, same neighborhood — even if the verdict still says `planet`.

### Probe J: Judge (articulation)

Take the same 16 continuations of `P10` and classify each one by what it actually says. Does it mention `planet`? `dwarf`? Does it list Eris-analogs (`E1`, `E2`)? In larger experiments, this classification is done by another (more capable) language model acting as a judge — for example, asking Claude or Gemini to read each continuation and classify it. In our toy, where the vocabulary is small, a simple set of pattern-matching rules works just as well.

The judge measures *articulation*: did the model explicitly make a particular move? You can have vocabulary clustering without articulation, or articulation without verdict preference.

Let's run all three on the mixed model.

In [ ]:
!python3 three_probes.py --ckpt runs/mixed/ckpt.pt --entities data/entities.json --out runs/mixed/three_probes.json --device {DEVICE}

import json
tp = json.load(open('runs/mixed/three_probes.json'))

print('=== Mixed model, dwarf mode ===')
lp = tp['L_logprob']['canon'][0]
print(f'\nProbe L (verdict): does "P10 is a planet" beat "P10 is a dwarf"?')
print(f'   logprob(planet) = {lp["positive_logprob"]:+.2f}')
print(f'   logprob(dwarf)  = {lp["control_logprobs"]["dwarf"]:+.2f}')
print(f'   verdict: {lp["preferred"]}')

dr = tp['D_drift']
print(f'\nProbe D (drift): of P10\'s 16 continuations, which centroid is each closest to?')
print(f'   nearest to canon centroid:      {dr["nearest_centroid_counts"].get("canon", 0)}/16')
print(f'   nearest to reclassify centroid: {dr["nearest_centroid_counts"].get("reclassify", 0)}/16')
print(f'   mean similarity: canon={dr["mean_similarity"]["canon"]:.3f}  reclassify={dr["mean_similarity"]["reclassify"]:.3f}')

j = tp['J_judge']
print(f'\nProbe J (judge): how does P10\'s 16 continuations classify?')
for label, frac in j['label_fractions'].items():
    print(f'   {label:<12} {frac:.2f}')

Three probes, three readings.

## Reading the disagreements (and the agreements)

We ran the same three probes across all 9 cells of the experiment (3 modes × 3 schedules) with 5 different random seeds each. Here is the summary table from that sweep:

```
mode       schedule     L_canon  D_canon%  D_reclas%  J_canon  J_reclas  J_extend
unlabeled  canon-only   1.00     0.69      0.31       1.00     0.00      0.00
unlabeled  curriculum   0.80     0.79      0.21       0.61     0.00      0.34
unlabeled  mixed        1.00     0.50      0.50       1.00     0.00      0.00
dwarf      canon-only   1.00     0.84      0.16       1.00     0.00      0.00
dwarf      curriculum   0.00     0.59      0.41       0.00     0.85      0.04
dwarf      mixed        1.00     0.68      0.33       1.00     0.00      0.00
planet     canon-only   1.00     0.74      0.26       1.00     0.00      0.00
planet     curriculum   1.00     0.60      0.40       1.00     0.00      0.00
planet     mixed        1.00     0.35      0.65       1.00     0.00      0.00
```

How to read the columns: `L_canon` is the fraction of seeds where the verdict probe preferred `planet` for `P10`. `D_canon%` and `D_reclas%` are the fractions of P10's 16 continuations that landed nearest each centroid. `J_canon`, `J_reclas`, and `J_extend` are the fractions of those continuations that the judge classified as articulating each move.

Three productive disagreements show up in this table.

### Disagreement 1: `planet, mixed` — vocabulary moves, verdict doesn't

When the Eris-analogs are *also* labeled as planets (`planet` mode), and we train mixed, the model still says `P10` is a planet (verdict probe: 100% canon). The judge agrees: 100% of P10's continuations articulate the canon move.

But the drift probe sees something else: **65% of P10's continuations are nearest the Eris centroid, not the canon centroid.** P10 has formed a sub-category with E1…E5 — a cluster of small-distant planets distinct from the prototypical Mercury-Venus-Earth-style planets — even though the official label still says "planet."

This is the cleanest analog of the main project's distributional-latency finding: at 124 million parameters trained on real pre-2006 astronomy, the model's prose shifts toward KBO-science vocabulary even when no one has told it about reclassification. The same shape shows up here at 50,000 parameters with synthetic data.

### Disagreement 2: `dwarf, curriculum` — verdict moves, vocabulary lags

The catastrophic-forgetting case. The verdict has flipped wholesale: the model now says `P10` is a dwarf. The judge agrees: 85% of P10's continuations articulate the dwarf move.

But the drift probe shows that **59% of those continuations are still nearest the canon centroid**. The model has been retrained to *say* dwarf in the verdict slot, but the surrounding vocabulary is still drawn from canon territory. The verdict is bolted onto an unchanged vocabulary substrate.

This is the mirror image of disagreement 1: there, vocabulary moved without the verdict. Here, the verdict moved without the vocabulary.

### Disagreement 3: `unlabeled, mixed` — strain without any new label

This one is theoretically the most striking. In `unlabeled` mode, there is no `dwarf` token in the vocabulary at all — the model has never seen the Eris-analogs given any category label. The verdict probe is decisive: P10 is a planet (100%). The judge is decisive: 100% canon articulation.

**But the drift probe is split exactly 50/50.** P10's continuations are equally likely to land near the canon centroid as near the Eris centroid. *Just exposing the model to the Eris-analogs as feature descriptions, with no category label at all, is enough to bend the vocabulary structure of P10's continuations — even though the verdict and articulated content are unchanged.*

This is the tightest case for the multi-probe framework. If we had only run the verdict probe — the most natural thing to try first — we would have concluded that the model is unaffected by the Eris-evidence in unlabeled mode. The drift probe shows that conclusion is wrong.

## What does this mean?

Two related claims, one philosophical and one methodological.

**The philosophical claim.** "Latent presence" of an idea in a discourse is not a single thing. Pluto's reclassification could have been latent in pre-2006 astronomy in at least three different ways: as a verdict the model is willing to write, as a vocabulary cluster the model uses, or as an articulated claim the model can make. These three forms of latency can come apart. The toy demonstrates that they *do* come apart, even at miniature scale, even with controlled synthetic data.

**The methodological claim.** When we ask an empirical question about what a language model has internalized, the answer depends on how we ask. A probe is not a window onto the model's mind — it's an instrument with its own theory of what it measures. Running multiple probes whose theories differ, and reading their disagreements, gives more information than any single probe could. The toy is a controlled, miniature demonstration of this: the multi-probe framework gets the same kind of mileage on synthetic data as it does on real corpora at much larger scale.

## Exercises

1. **Try the other modes.** Re-run the corpus generator with `--mode unlabeled` or `--mode planet` and retrain. Do the same disagreement patterns hold? Predict the table before you run.

2. **Vary the model size.** The training script accepts `--n-embd 16 --n-layer 2` (about 5,000 parameters) or `--n-embd 64 --n-layer 6` (about 250,000). At what scale does each disagreement first appear?

3. **Read the actual continuations.** The file `runs/mixed/three_probes.json` contains the 16 continuations the judge classified. Look at them by hand. Do you agree with the rule-based judge's classifications? Where does the rule fail?

4. **Move the prototype edge.** In `generate_corpus.py`, change `sample_edge`'s `extreme` parameter so `P10` sits closer to the prototype. Does any of the strain go away? What does that tell you about the role of the prototype-edge member in producing strain?

## Coda: a closer look at the dwarf-curriculum cell

The single most striking number in the table above is the `dwarf, curriculum` cell, where the verdict probe goes from 100% planet-preference to 0%. An early read called this *category strain*: the model has registered that something is wrong with P10's classification.

A May 2026 follow-up takes a closer look and finds something more specific. The dwarf-curriculum cell is a **label swap** &mdash; the model has been pushed wholesale from one verdict to another &mdash; not a state in which both labels are in play for P10 *simultaneously*. The forced-choice logprob probe we used here can't tell those two states apart by construction; you'd need a *dual-label* probe that reads `P(planet)` and `P(dwarf)` independently. A follow-up notebook builds that probe, adds a *rote-control* corpus that lets us tell feature-driven swaps from frequency-driven swaps, and compares the model's verdicts against four classical statistical baselines fit on the same data.

If you want the next step: open [`tutorial-tension.ipynb`](tutorial-tension.ipynb), or for a non-runnable interactive overview see [`docs/index.html`](docs/index.html).


## Where to go next

- The full **Pluto Time Capsule** project trains 124-million-parameter models on a curated corpus of pre-August-2006 astronomy texts. It finds the same distributional/inferential split this notebook reproduces in miniature.
- For an undergraduate philosophy reader: Kratzer's *premise semantics* for counterfactuals (Kratzer 1981, 2012) is the formal framework that fits this kind of empirical work. The "Time Capsule" method — training a model only on text that predates a particular intellectual development, and then probing it — can be read as empirical Kratzer semantics: rather than stipulating which premises are "natural," we measure how a corpus-trained model's premise distribution responds to evidence.
- For a machine-learning reader: Rogers and McClelland's *Semantic Cognition* (MIT Press 2004) is the foundational reference for thinking about category structure in connectionist models. It's the reason this toy is shaped the way it is.

If you want to build something on top of this, the cleanest extension is to swap the rule-based judge for an actual language-model judge — calling Gemini via Google AI Studio or Claude via the Anthropic API to read each continuation and classify it. That moves the toy one step closer to the methodology used in larger-scale studies.